# NEMATICA human test: reduce the open CCEP ECoG dataset in the cloud

This notebook downloads every recording of OpenNeuro **ds004080** (van Blooijs et al. 2023, CC0; about 289 GB) **one file at a time**, reduces each to a small summary with `scripts/ccep_extract.py` (average responses to each pulse, and the statistics needed to fit watching copies), saves the summary to your Google Drive, and deletes the raw file before fetching the next. Peak disk use is one recording (at most about 3 GB).

**Run:** Runtime > Run all. Allow Google Drive access when asked. It can be stopped and restarted at any time: recordings already in `MyDrive/nematica_ccep` are skipped. Expect a few hours. When it finishes, download the `nematica_ccep` folder (a few hundred MB).

The analysis itself, and the predictions it tests (written before any data was downloaded), are in the project README: *Beyond the worm: the same test in human brains*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/nematica_ccep'
import os
os.makedirs(OUT, exist_ok=True)
!git clone -q --depth 1 https://github.com/OpenNeuroDatasets/ds004080.git /content/ds004080 || true
!wget -q -O /content/ccep_extract.py https://raw.githubusercontent.com/Weptune/nematica/main/scripts/ccep_extract.py
!cp /content/ds004080/participants.tsv "$OUT/" && df -h /content | tail -1

In [ ]:
import glob, sys, time, traceback
import numpy as np, requests
sys.path.insert(0, '/content')
import ccep_extract as E

ROOT = '/content/ds004080/'
BASE = 'https://s3.amazonaws.com/openneuro.org/ds004080/'

def fetch(rel):
    dst = ROOT + rel
    if os.path.lexists(dst) and not os.path.islink(dst) and os.path.getsize(dst) > 0:
        return
    if os.path.lexists(dst):
        os.remove(dst)
    for attempt in range(5):
        try:
            with requests.get(BASE + rel, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(dst + '.part', 'wb') as f:
                    for chunk in r.iter_content(1 << 24):
                        f.write(chunk)
            os.replace(dst + '.part', dst)
            return
        except Exception as e:
            print('  retry', attempt + 1, rel.split('/')[-1], e)
            time.sleep(10 * (attempt + 1))
    raise RuntimeError('could not download ' + rel)

runs = sorted(glob.glob(ROOT + 'sub-*/ses-*/ieeg/*_ieeg.vhdr'))
print(len(runs), 'recordings')
log = open(OUT + '/log.txt', 'a')
for i, v in enumerate(runs):
    stem = os.path.basename(v).replace('_ieeg.vhdr', '')
    out = f'{OUT}/{stem}.npz'
    if os.path.exists(out):
        continue
    rel = os.path.relpath(v, ROOT)
    t0 = time.time()
    try:
        for ext in ('_ieeg.vhdr', '_ieeg.vmrk', '_ieeg.eeg'):
            fetch(rel.replace('_ieeg.vhdr', ext))
        res = E.process(__import__('pathlib').Path(v))
        np.savez_compressed(out + '.part.npz', **res)
        os.replace(out + '.part.npz', out)
        msg = (f"{i+1}/{len(runs)} {stem}: {len(res['names'])} channels, {len(res['sites'])} sites, "
               f"{int(res['n_odd'].sum() + res['n_even'].sum())} pulses, {res['seg_seconds']:.0f} s between pulses, "
               f"{time.time() - t0:.0f} s")
    except Exception:
        msg = f'{i+1}/{len(runs)} {stem}: FAILED\n' + traceback.format_exc()
    finally:
        eeg = ROOT + rel.replace('_ieeg.vhdr', '_ieeg.eeg')
        if os.path.exists(eeg) and not os.path.islink(eeg):
            os.remove(eeg)
    print(msg); log.write(msg + '\n'); log.flush()
print('done')

In [ ]:
# summary of what was saved
done = sorted(glob.glob(OUT + '/*.npz'))
print(len(done), 'of', len(runs), 'recordings reduced;',
      round(sum(os.path.getsize(p) for p in done) / 1e6), 'MB in', OUT)
!grep -c FAILED "$OUT/log.txt" || true